# 01 · Training queue (resumable)
Runs the experiment plan from the Stage-1 Methodology Blueprint. **If Colab disconnects, just run all cells again** — finished runs are skipped and the interrupted run resumes from its last checkpoint (saved every 500 iterations on Drive).

You can open this notebook in two Colab sessions at once (e.g. different filters); a lock file prevents two sessions from training the same run.

| SQ | Split | Backbones | Losses | Seeds | Runs |
|---|---|---|---|---|---|
| SQ1 consistency | random (stratified by institute) | 4 | base / clDice / Skeleton Recall | 3 | 36 |
| SQ2 cross-institute | region (GWFSS paper) | 4 | 3 | 3 | 36 |
| SQ3 resolution | random, input ×2 | SegFormer-B1, DeepLabV3+ | 3 | 3 | 18 |

All arms share identical hyperparameters (8,000 iterations, batch 8, crop 512, AdamW 1e-4, poly schedule, AMP). Only the loss differs. The final iterate is reported (no checkpoint selection on validation).

In [ ]:
# --- Setup (run every session) ---
from google.colab import drive
drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/gwfss_stem_seg'
DATA_ROOT = '/content/gwfss'                      # local disk: fast I/O, rebuilt each session
SPLIT_FILE = f'{PROJECT}/splits/splits.json'      # frozen once, shared by all runs
RESULTS = f'{PROJECT}/results'                    # checkpoints, logs, metrics (on Drive)

import sys, os
sys.path.insert(0, f'{PROJECT}/src')
!pip -q install segmentation-models-pytorch==0.5.0
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from gwfss_stem import data
manifest = data.prepare_dataset(DATA_ROOT)   # downloads GWFSS_v1.0 from Hugging Face (~600 MB) if not present
print(len(manifest), 'images')

In [ ]:
from gwfss_stem import plan, engine
import pandas as pd

# --- choose what this session should run ---
FILTER = dict(sq='SQ1')                 # e.g. dict(sq='SQ1', backbone='segformer_b1') or dict(sq=['SQ1','SQ2'])
OVERRIDES = dict(batch_size=8)          # reduce batch_size only if you hit out-of-memory (then keep it for ALL runs)

runs = plan.select(plan.full_plan(), **FILTER)
def status(r):
    d = os.path.join(RESULTS, engine.run_name({**engine.DEFAULTS, **r}))
    if os.path.exists(f'{d}/final.pt'): return 'done'
    if os.path.exists(f'{d}/RUNNING'): return 'running/locked'
    if os.path.exists(f'{d}/last.pt'): return 'partial'
    return 'pending'
table = pd.DataFrame([{**r, 'status': status(r)} for r in runs])
display(table.status.value_counts()); display(table)

In [ ]:
for r in runs:
    cfg = {k: v for k, v in r.items() if k != 'sq'}
    cfg.update(OVERRIDES)
    run_dir = engine.train_run(cfg, DATA_ROOT, SPLIT_FILE, RESULTS)
    if run_dir:
        engine.evaluate_run(run_dir, DATA_ROOT, SPLIT_FILE, subset='test', tag='ss')